In [1]:
pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import openpyxl 


In [3]:
behave = pd.read_csv("/Users/riya/Downloads/QVI_purchase_behaviour.csv")
trans = pd.read_excel("/Users/riya/Downloads/QVI_transaction_data.xlsx", dtype={'DATE': 'int64'})

print(behave.head())
print(trans.head())

   LYLTY_CARD_NBR               LIFESTAGE PREMIUM_CUSTOMER
0            1000   YOUNG SINGLES/COUPLES          Premium
1            1002   YOUNG SINGLES/COUPLES       Mainstream
2            1003          YOUNG FAMILIES           Budget
3            1004   OLDER SINGLES/COUPLES       Mainstream
4            1005  MIDAGE SINGLES/COUPLES       Mainstream
    DATE  STORE_NBR  LYLTY_CARD_NBR  TXN_ID  PROD_NBR  \
0  43390          1            1000       1         5   
1  43599          1            1307     348        66   
2  43605          1            1343     383        61   
3  43329          2            2373     974        69   
4  43330          2            2426    1038       108   

                                  PROD_NAME  PROD_QTY  TOT_SALES  
0    Natural Chip        Compny SeaSalt175g         2        6.0  
1                  CCs Nacho Cheese    175g         3        6.3  
2    Smiths Crinkle Cut  Chips Chicken 170g         2        2.9  
3    Smiths Chip Thinly  S/Cream&On

In [4]:
behave.isnull()
trans.isnull()

,DATE,STORE_NBR,LYLTY_CARD_NBR,TXN_ID,PROD_NBR,PROD_NAME,PROD_QTY,TOT_SALES
0,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...
264831,False,False,False,False,False,False,False,False
264832,False,False,False,False,False,False,False,False
264833,False,False,False,False,False,False,False,False
264834,False,False,False,False,False,False,False,False


In [5]:
trans = trans[trans["PROD_QTY"]> 0]

In [6]:
trans['DATE'] = pd.to_datetime(trans['DATE'], origin='1899-12-30', unit='D')


In [7]:
print(trans.head())

        DATE  STORE_NBR  LYLTY_CARD_NBR  TXN_ID  PROD_NBR  \
0 2018-10-17          1            1000       1         5   
1 2019-05-14          1            1307     348        66   
2 2019-05-20          1            1343     383        61   
3 2018-08-17          2            2373     974        69   
4 2018-08-18          2            2426    1038       108   

                                  PROD_NAME  PROD_QTY  TOT_SALES  
0    Natural Chip        Compny SeaSalt175g         2        6.0  
1                  CCs Nacho Cheese    175g         3        6.3  
2    Smiths Crinkle Cut  Chips Chicken 170g         2        2.9  
3    Smiths Chip Thinly  S/Cream&Onion 175g         5       15.0  
4  Kettle Tortilla ChpsHny&Jlpno Chili 150g         3       13.8  


In [8]:
# Extract brand name (first word) from PROD_NAME
trans['BRAND'] = trans['PROD_NAME'].str.split().str[0]

# Extract packet size (e.g., 175g) from the end of PROD_NAME
trans['PACKET_SIZE'] = trans['PROD_NAME'].str.extract(r'(\d+g)$')

print(trans[['PROD_NAME', 'BRAND', 'PACKET_SIZE']].head())

                                  PROD_NAME    BRAND PACKET_SIZE
0    Natural Chip        Compny SeaSalt175g  Natural        175g
1                  CCs Nacho Cheese    175g      CCs        175g
2    Smiths Crinkle Cut  Chips Chicken 170g   Smiths        170g
3    Smiths Chip Thinly  S/Cream&Onion 175g   Smiths        175g
4  Kettle Tortilla ChpsHny&Jlpno Chili 150g   Kettle        150g


In [9]:
# Filter products that contain 'chip' in PROD_NAME (case insensitive)
chips_trans = trans[trans['PROD_NAME'].str.contains('chip', case=False)]
print(chips_trans.head())

        DATE  STORE_NBR  LYLTY_CARD_NBR  TXN_ID  PROD_NBR  \
0 2018-10-17          1            1000       1         5   
2 2019-05-20          1            1343     383        61   
3 2018-08-17          2            2373     974        69   
6 2019-05-16          4            4149    3333        16   
8 2018-08-20          5            5026    4525        42   

                                  PROD_NAME  PROD_QTY  TOT_SALES    BRAND  \
0    Natural Chip        Compny SeaSalt175g         2        6.0  Natural   
2    Smiths Crinkle Cut  Chips Chicken 170g         2        2.9   Smiths   
3    Smiths Chip Thinly  S/Cream&Onion 175g         5       15.0   Smiths   
6  Smiths Crinkle Chips Salt & Vinegar 330g         1        5.7   Smiths   
8   Doritos Corn Chip Mexican Jalapeno 150g         1        3.9  Doritos   

  PACKET_SIZE  
0        175g  
2        170g  
3        175g  
6        330g  
8        150g  


In [10]:
# Calculate unit price by dividing TOT_SALES by PROD_QTY
chips_trans['UNIT_PRICE'] = chips_trans['TOT_SALES'] / chips_trans['PROD_QTY']
print(chips_trans[['PROD_NAME', 'TOT_SALES', 'PROD_QTY', 'UNIT_PRICE']].head())

                                  PROD_NAME  TOT_SALES  PROD_QTY  UNIT_PRICE
0    Natural Chip        Compny SeaSalt175g        6.0         2        3.00
2    Smiths Crinkle Cut  Chips Chicken 170g        2.9         2        1.45
3    Smiths Chip Thinly  S/Cream&Onion 175g       15.0         5        3.00
6  Smiths Crinkle Chips Salt & Vinegar 330g        5.7         1        5.70
8   Doritos Corn Chip Mexican Jalapeno 150g        3.9         1        3.90


/var/folders/3c/hz1h3tqj7kg622xk2kz_8ys40000gn/T/ipykernel_11292/1403124054.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chips_trans['UNIT_PRICE'] = chips_trans['TOT_SALES'] / chips_trans['PROD_QTY']


In [11]:
# Merge chips_trans with behave on LYLTY_CARD_NBR
merged = pd.merge(chips_trans, behave, on='LYLTY_CARD_NBR')

# 1. Total Sales by Segment
total_sales = merged.groupby(['LIFESTAGE', 'PREMIUM_CUSTOMER'])['TOT_SALES'].sum().sort_values(ascending=False)
print("Total Sales by Segment:")
print(total_sales.head())
print()

# 2. Number of Customers by Segment
num_customers = merged.groupby(['LIFESTAGE', 'PREMIUM_CUSTOMER'])['LYLTY_CARD_NBR'].nunique().sort_values(ascending=False)
print("Number of Customers by Segment:")
print(num_customers.head())
print()

# 3. Average Units per Customer (total PROD_QTY / num customers)
total_qty = merged.groupby(['LIFESTAGE', 'PREMIUM_CUSTOMER'])['PROD_QTY'].sum()
avg_units_per_customer = (total_qty / num_customers).sort_values(ascending=False)
print("Average Units per Customer:")
print(avg_units_per_customer.head())
print()

# 4. Average Price per Unit by Segment
avg_price_per_unit = merged.groupby(['LIFESTAGE', 'PREMIUM_CUSTOMER'])['UNIT_PRICE'].mean().sort_values(ascending=False)
print("Average Price per Unit by Segment:")
print(avg_price_per_unit.head())

Total Sales by Segment:
LIFESTAGE              PREMIUM_CUSTOMER
OLDER FAMILIES         Budget              44859.2
RETIREES               Mainstream          40592.1
YOUNG SINGLES/COUPLES  Mainstream          40069.9
YOUNG FAMILIES         Budget              37064.1
OLDER SINGLES/COUPLES  Budget              35943.0
Name: TOT_SALES, dtype: float64

Number of Customers by Segment:
LIFESTAGE              PREMIUM_CUSTOMER
YOUNG SINGLES/COUPLES  Mainstream          3952
RETIREES               Mainstream          3798
OLDER FAMILIES         Budget              3177
OLDER SINGLES/COUPLES  Budget              3114
                       Mainstream          3062
Name: LYLTY_CARD_NBR, dtype: int64

Average Units per Customer:
LIFESTAGE       PREMIUM_CUSTOMER
OLDER FAMILIES  Mainstream          4.080957
                Budget              3.998426
                Premium             3.984987
YOUNG FAMILIES  Premium             3.919463
                Budget              3.893562
dtype: float64

In [12]:
# Data Cleanup: Standardize brand names
brand_mapping = {
    'Natural': 'Natural',
    'Smiths': 'Smiths',
    'Kettle': 'Kettle',
    'Doritos': 'Doritos',
    'Pringles': 'Pringles',
    'Infuzions': 'Infuzions',
    'Thins': 'Thins',
    'CCs': 'CCs',
    'Woolworths': 'Woolworths',
    'Cheezels': 'Cheezels',
    'Tostitos': 'Tostitos',
    'Twisties': 'Twisties',
    'Grain': 'Grain',
    'Cheetos': 'Cheetos',
    'Burger': 'Burger',
    'Tyrrells': 'Tyrrells',
    'Cobs': 'Cobs',
    'French': 'French',
    'RRD': 'RRD',
    'Sunbites': 'Sunbites',
    'Woolworths': 'Woolworths',
    'Snbts': 'Sunbites',  # Assuming abbreviation
    'Infzns': 'Infuzions',
    'Red': 'Red',
    'WW': 'Woolworths',
    'Smith': 'Smiths',  # Merge Smith to Smiths
    'NCC': 'Natural',  # Assuming NCC is Natural Chip Company
    'Dorito': 'Doritos',
    'GrnWves': 'Grain Waves',
    'Natural Chip': 'Natural'  # If any
}
merged['BRAND_CLEAN'] = merged['BRAND'].map(brand_mapping).fillna(merged['BRAND'])

print("Unique cleaned brands:")
print(merged['BRAND_CLEAN'].unique())

Unique cleaned brands:
['Natural' 'Smiths' 'Doritos' 'Thins' 'Cobs' 'French' 'Woolworths'
 'Tostitos']


In [13]:
# Brand Affinity: Compare brands for Mainstream Young Singles vs. others
# Calculate brand sales proportion for each segment
brand_sales = merged.groupby(['LIFESTAGE', 'PREMIUM_CUSTOMER', 'BRAND_CLEAN'])['TOT_SALES'].sum().reset_index()
total_sales_segment = merged.groupby(['LIFESTAGE', 'PREMIUM_CUSTOMER'])['TOT_SALES'].sum().reset_index()
brand_sales = pd.merge(brand_sales, total_sales_segment, on=['LIFESTAGE', 'PREMIUM_CUSTOMER'])
brand_sales['PROPORTION'] = brand_sales['TOT_SALES_x'] / brand_sales['TOT_SALES_y']

# Filter for Mainstream Young Singles
mainstream_young = brand_sales[(brand_sales['LIFESTAGE'] == 'YOUNG SINGLES/COUPLES') & (brand_sales['PREMIUM_CUSTOMER'] == 'Mainstream')].sort_values('PROPORTION', ascending=False)
print("Top brands for Mainstream Young Singles:")
print(mainstream_young[['BRAND_CLEAN', 'PROPORTION']].head())
print()

# Compare to overall or another segment, e.g., Budget Young Singles
budget_young = brand_sales[(brand_sales['LIFESTAGE'] == 'YOUNG SINGLES/COUPLES') & (brand_sales['PREMIUM_CUSTOMER'] == 'Budget')].sort_values('PROPORTION', ascending=False)
print("Top brands for Budget Young Singles:")
print(budget_young[['BRAND_CLEAN', 'PROPORTION']].head())

Top brands for Mainstream Young Singles:
    BRAND_CLEAN  PROPORTION
153     Doritos    0.293300
156      Smiths    0.235968
157       Thins    0.180113
152        Cobs    0.153347
158    Tostitos    0.056881

Top brands for Budget Young Singles:
    BRAND_CLEAN  PROPORTION
148      Smiths    0.254436
145     Doritos    0.228693
149       Thins    0.166995
144        Cobs    0.119592
147     Natural    0.087619


In [14]:
# Pack Size Trends: Compare 330g purchases for Older Families vs. Young Singles
pack_size_counts = merged.groupby(['LIFESTAGE', 'PREMIUM_CUSTOMER', 'PACKET_SIZE'])['PROD_QTY'].sum().reset_index()

# Filter for 330g
pack_330 = pack_size_counts[pack_size_counts['PACKET_SIZE'] == '330g']

# Older Families
older_families = pack_330[pack_330['LIFESTAGE'] == 'OLDER FAMILIES']
print("330g purchases by Older Families segments:")
print(older_families[['PREMIUM_CUSTOMER', 'PROD_QTY']])
print()

# Young Singles
young_singles = pack_330[pack_330['LIFESTAGE'] == 'YOUNG SINGLES/COUPLES']
print("330g purchases by Young Singles segments:")
print(young_singles[['PREMIUM_CUSTOMER', 'PROD_QTY']])

# Overall comparison
total_330_older = older_families['PROD_QTY'].sum()
total_330_young = young_singles['PROD_QTY'].sum()
print(f"\nTotal 330g units: Older Families: {total_330_older}, Young Singles: {total_330_young}")

330g purchases by Older Families segments:
   PREMIUM_CUSTOMER  PROD_QTY
54           Budget       559
62       Mainstream       333
70          Premium       257

330g purchases by Young Singles segments:
    PREMIUM_CUSTOMER  PROD_QTY
150           Budget       170
158       Mainstream       582
166          Premium       113

Total 330g units: Older Families: 1149, Young Singles: 865


In [15]:
# Market Share & Relative Spend
total_chip_sales = merged['TOT_SALES'].sum()

# Market Share %
market_share = (total_sales / total_chip_sales * 100).sort_values(ascending=False)
print("Market Share % by Segment:")
print(market_share)
print()

# Sales per Customer
sales_per_customer = (total_sales / num_customers).sort_values(ascending=False)
print("Sales per Customer by Segment:")
print(sales_per_customer)
print()

# Trips per Customer (num transactions / num customers)
num_trips = merged.groupby(['LIFESTAGE', 'PREMIUM_CUSTOMER'])['TXN_ID'].nunique()
trips_per_customer = (num_trips / num_customers).sort_values(ascending=False)
print("Trips per Customer by Segment:")
print(trips_per_customer)
print()

# Packets per Trip
packets_per_trip = merged.groupby(['LIFESTAGE', 'PREMIUM_CUSTOMER', 'TXN_ID'])['PROD_QTY'].sum().groupby(level=[0,1]).mean().sort_values(ascending=False)
print("Average Packets per Trip by Segment:")
print(packets_per_trip)

Market Share % by Segment:
LIFESTAGE               PREMIUM_CUSTOMER
OLDER FAMILIES          Budget              8.768613
RETIREES                Mainstream          7.934525
YOUNG SINGLES/COUPLES   Mainstream          7.832450
YOUNG FAMILIES          Budget              7.244908
OLDER SINGLES/COUPLES   Budget              7.025767
                        Mainstream          6.928071
                        Premium             6.752500
RETIREES                Budget              5.874216
OLDER FAMILIES          Mainstream          5.531499
YOUNG FAMILIES          Mainstream          4.949194
RETIREES                Premium             4.848508
MIDAGE SINGLES/COUPLES  Mainstream          4.681577
YOUNG FAMILIES          Premium             4.453014
OLDER FAMILIES          Premium             4.154923
YOUNG SINGLES/COUPLES   Budget              3.279414
MIDAGE SINGLES/COUPLES  Premium             3.000342
YOUNG SINGLES/COUPLES   Premium             2.267331
MIDAGE SINGLES/COUPLES  Budget 

In [16]:
# Statistical Significance: T-test for UNIT_PRICE difference
from scipy.stats import ttest_ind

# Mainstream Young Singles
mainstream_young_prices = merged[(merged['LIFESTAGE'] == 'YOUNG SINGLES/COUPLES') & (merged['PREMIUM_CUSTOMER'] == 'Mainstream')]['UNIT_PRICE']

# Compare to Budget Young Singles
budget_young_prices = merged[(merged['LIFESTAGE'] == 'YOUNG SINGLES/COUPLES') & (merged['PREMIUM_CUSTOMER'] == 'Budget')]['UNIT_PRICE']

t_stat, p_value = ttest_ind(mainstream_young_prices, budget_young_prices)
print(f"T-test Mainstream vs Budget Young Singles: t-stat={t_stat:.2f}, p-value={p_value:.4f}")
if p_value < 0.05:
    print("Significant difference in unit prices.")
else:
    print("No significant difference.")

# Compare to Premium Young Singles
premium_young_prices = merged[(merged['LIFESTAGE'] == 'YOUNG SINGLES/COUPLES') & (merged['PREMIUM_CUSTOMER'] == 'Premium')]['UNIT_PRICE']
t_stat2, p_value2 = ttest_ind(mainstream_young_prices, premium_young_prices)
print(f"T-test Mainstream vs Premium Young Singles: t-stat={t_stat2:.2f}, p-value={p_value2:.4f}")
if p_value2 < 0.05:
    print("Significant difference.")
else:
    print("No significant difference.")

T-test Mainstream vs Budget Young Singles: t-stat=17.58, p-value=0.0000
Significant difference in unit prices.
T-test Mainstream vs Premium Young Singles: t-stat=14.93, p-value=0.0000
Significant difference.


In [17]:
# Visualizations
import plotly.express as px
import plotly.graph_objects as go

# 1. Heatmap of Total Sales by Segment
sales_pivot = total_sales.reset_index().pivot(index='LIFESTAGE', columns='PREMIUM_CUSTOMER', values='TOT_SALES').fillna(0)
fig1 = go.Figure(data=go.Heatmap(
    z=sales_pivot.values,
    x=sales_pivot.columns,
    y=sales_pivot.index,
    colorscale='Blues'
))
fig1.update_layout(title='Total Sales by Segment', xaxis_title='Premium Customer', yaxis_title='Lifestage')
fig1.show()

# 2. Scatter Plot: Number of Customers vs Total Spend
customers_spend = pd.DataFrame({'num_customers': num_customers, 'total_sales': total_sales}).reset_index()
fig2 = px.scatter(customers_spend, x='num_customers', y='total_sales', color='LIFESTAGE', symbol='PREMIUM_CUSTOMER',
                  title='Number of Customers vs Total Sales by Segment')
fig2.show()

# 3. Distribution Plot of Pack Sizes for Top 3 Segments
top_segments = total_sales.head(3).index
top_data = merged[merged.set_index(['LIFESTAGE', 'PREMIUM_CUSTOMER']).index.isin(top_segments)]
fig3 = px.histogram(top_data, x='PACKET_SIZE', color='LIFESTAGE', facet_col='PREMIUM_CUSTOMER',
                    title='Pack Size Distribution for Top 3 Segments')
fig3.show()

In [18]:
# Commercial Recommendation
print("Commercial Recommendations:")
print("1. Target Older Families (Budget) with promotions on large packs (330g) as they have high sales and prefer family sizes.")
print("2. For Mainstream Young Singles, promote premium brands like Doritos and Thins, as they pay higher prices.")
print("3. Increase marketing to segments with high customer count but lower spend, like Young Singles Mainstream, by offering bundles.")
print("4. Focus on high-margin segments: Older Families Budget for volume, Young Singles Mainstream for premium pricing.")
print("5. Monitor pack size preferences: Older Families buy more 330g, so stock accordingly.")

Commercial Recommendations:
1. Target Older Families (Budget) with promotions on large packs (330g) as they have high sales and prefer family sizes.
2. For Mainstream Young Singles, promote premium brands like Doritos and Thins, as they pay higher prices.
3. Increase marketing to segments with high customer count but lower spend, like Young Singles Mainstream, by offering bundles.
4. Focus on high-margin segments: Older Families Budget for volume, Young Singles Mainstream for premium pricing.
5. Monitor pack size preferences: Older Families buy more 330g, so stock accordingly.
